In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pylab as plt
import pickle
import sklearn
from sklearn import metrics

from sklearn.manifold import TSNE
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score, recall_score, roc_auc_score

RandomSeed = 42
np.random.seed(RandomSeed)

sns.set(font_scale = 1.4)
sns.set_style("white")
sns.set_style("ticks", {"xtick.major.size": 6, "ytick.major.size": 0})

pd.set_option("display.max_colwidth", False)
pd.set_option('display.expand_frame_repr', False)
sns.set(font_scale = 1.2)

In [ ]:
def feature_extraction(df, output_filename):
    # extract feature from a df, utilize the deephase function in deephase_utils.py
    for row in df.iterrow():
        seq = row["row"]
        # extract feature here
        # save the feature combine with the original df

    # save df into a file
    df.to_csv(output_filename)
    return df

In [ ]:
def train_model(df_training, training_categories, num_trials, model_name, clf):
    df_training = df_training[df_training['Category'].isin(training_categories)]
    df_training['Category'] = df_training['Category'].map({str(training_categories[0]): 1, training_categories[1]: 0})
    df_training = df_training.reset_index(drop=True)
    df_training = df_training.reindex(sorted(df_training.columns), axis=1)

    gss = GroupShuffleSplit(n_splits = num_trials, test_size = 0.1, random_state=42)
    
    y = df_training['Category']
    groups = df_training['Uniprot_ID']
    X = df_training.drop(columns={'Category', 'Uniprot_ID', 'Sequence'})
    
    accuracy = []; precision = []
    recall = []; roc_auc = []
    for train_index, test_index in gss.split(X, y, groups=groups):
        clf.fit(X.loc[train_index], y.loc[train_index])
        score = clf.score(X.loc[test_index], y.loc[test_index])
        accuracy.append(accuracy_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        precision.append(precision_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        recall.append(recall_score(clf.predict(X.loc[test_index]), y.loc[test_index]))
        roc_auc.append(roc_auc_score(y.loc[test_index], clf.predict_proba(X.loc[test_index])[:,1]))
    print('Random forest model: accuracy =' + str(sum(accuracy)/num_trials) + '+/-' + str(np.std(accuracy)/np.sqrt(num_trials)))
    print('Random forest model: precision =' + str(sum(precision)/num_trials) + '+/-' + str(np.std(precision)/np.sqrt(num_trials)))
    print('Random forest model: recall =' + str(sum(recall)/num_trials) + '+/-' + str(np.std(recall)/np.sqrt(num_trials)))
    print('Random forest model: ROC-AUC =' + str(sum(roc_auc)/num_trials) + '+/-' + str(np.std(roc_auc)/np.sqrt(num_trials)))
    
    clf.fit(X, y)
    pickle.dump(clf, open('Models/' + str(model_name) + '.sav', 'wb'))